# APIM Compartido — Provisión única

## Shared APIM Gateway lab

Este notebook crea el **único** Azure API Management que usarán los 3 labs:

- Lab 1 — Model Gateway
- Lab 2 — Foundry IQ
- Lab 3 — Access Controlling

No crea Foundry, ni modelos, ni APIs de inferencia — solo la instancia de APIM, Log Analytics y Application Insights. Cada lab registrará después su propia API dentro de este APIM (`/inference`, `/ai-search`, `/access-inference`), referenciándolo como `existing`.

**Este notebook se ejecuta UNA sola vez, antes de correr cualquiera de los 3 labs.** No lo vuelvas a correr salvo que quieras destruir y recrear el APIM compartido (lo que rompería a los 3 labs hasta que actualices sus referencias).

<a id='0'></a>
### 0️⃣ Initialize notebook variables

In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = "shared-apim-gateway-V2"
resource_group_name = "rg-shared-apim-gateway-V2"
resource_group_location = "swedencentral"

# APIM configuration
# "Developer" = SKU para pruebas/labs (sin SLA, capacidad fija en 1, el más económico).
# Para producción evalúa "Basicv2", "Standardv2" o "Premium" — ver README.md.
apim_sku = 'Developer'
apim_subscriptions_config = [
    {"name": "shared-subscription", "displayName": "Shared AI Gateway Subscription"}
]

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 15:51:39.544068 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 15:51:55.829588 :4s]
👉🏽 Current user: ycure@coem.co
👉🏽 Tenant ID: 0c67c9dd-067e-4ab7-adc3-eac91ce94463
👉🏽 Subscription ID: efbaff8f-21cc-49db-8141-2caaf996decd


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

Crea el Resource Group dedicado (`rg-shared-apim-gateway`) y despliega [main.bicep](main.bicep): Log Analytics, Application Insights y el APIM único.

In [3]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters, indent=2))

# Run the deployment
output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

⚙️ Running: az group show --name rg-shared-apim-gateway-V2 
👉🏽 Resource group rg-shared-apim-gateway-V2 does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name rg-shared-apim-gateway-V2 --location swedencentral --tags source=ai-gateway 
✅ Resource group 'rg-shared-apim-gateway-V2' created ⌚ 15:52:18.781775 :6s]
⚙️ Running: az deployment group create --name shared-apim-gateway-V2 --resource-group rg-shared-apim-gateway-V2 --template-file main.bicep --parameters params.json 
✅ Deployment 'shared-apim-gateway-V2' succeeded ⌚ 16:20:32.424327 :13s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Estos son los valores que necesitarás pegar en la configuración de **Lab 1, Lab 2 y Lab 3** para que cada uno referencie este APIM como `existing`.

In [4]:
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    shared_apim_resource_group = utils.get_deployment_output(output, 'resourceGroupName', 'Resource Group Name')
    shared_apim_name = utils.get_deployment_output(output, 'apimServiceName', 'APIM Service Name')
    shared_apim_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM Gateway URL')
    shared_apim_principal_id = utils.get_deployment_output(output, 'apimPrincipalId', 'APIM Principal Id')
    shared_apim_logger_id = utils.get_deployment_output(output, 'apimLoggerId', 'APIM Logger Id')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        utils.print_info(f"Subscription Name: {subscription['name']}")
        utils.print_info(f"Subscription Key: ****{subscription['key'][-4:]}")

    utils.print_ok("\n✅ Shared APIM Gateway listo")
    utils.print_info("Copia estos valores en la configuración de Lab 1, Lab 2 y Lab 3:")
    print(json.dumps({
        "shared_apim_resource_group": shared_apim_resource_group,
        "shared_apim_name": shared_apim_name,
        "shared_apim_gateway_url": shared_apim_gateway_url,
    }, indent=2))

⚙️ Running: az deployment group show --name shared-apim-gateway-V2 -g rg-shared-apim-gateway-V2 
✅ Retrieved deployment: shared-apim-gateway-V2 ⌚ 16:22:03.026516 :7s]
👉🏽 Resource Group Name: rg-shared-apim-gateway-V2
👉🏽 APIM Service Name: apim-shared-pdcibwky2f5ms
👉🏽 APIM Gateway URL: https://apim-shared-pdcibwky2f5ms.azure-api.net
👉🏽 APIM Principal Id: 944ee3f2-5dc4-446a-9507-0424cd3020e7
👉🏽 APIM Logger Id: /subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/providers/Microsoft.ApiManagement/service/apim-shared-pdcibwky2f5ms/loggers/azuremonitor
👉🏽 Subscription Name: shared-subscription
👉🏽 Subscription Key: ****dce8
✅ 
✅ Shared APIM Gateway listo ⌚ 16:22:03.026516 
👉🏽 Copia estos valores en la configuración de Lab 1, Lab 2 y Lab 3:
{
  "shared_apim_resource_group": "rg-shared-apim-gateway-V2",
  "shared_apim_name": "apim-shared-pdcibwky2f5ms",
  "shared_apim_gateway_url": "https://apim-shared-pdcibwky2f5ms.azure-api.net"
}


<a id='clean'></a>
### 🗑️ Clean up resources

⚠️ Solo elimina estos recursos si ya no vas a usar ninguno de los 3 labs — borrar el APIM compartido rompe a Lab 1, Lab 2 y Lab 3 hasta que se vuelva a desplegar y se actualicen sus referencias.

```bash
az group delete --name rg-shared-apim-gateway --yes --no-wait
```